In [12]:
import sys
import os

SCRIPT_DIR = os.path.dirname(os.path.abspath("__file__"))
sys.path.append(os.path.dirname(SCRIPT_DIR))

In [13]:
import os
import yaml
import mne
import os
import numpy as np
import pandas as pd
import random
from datetime import timedelta
import matplotlib
import matplotlib.pyplot  as plt
import matplotlib.dates as mdates
import PyQt5
from data.eeg_loader import load_eeg, load_stimulus, get_eeg_timestamps
from data.eeg_loader import trial_start_sec, trial_end_sec, detect_signal_start
from paradigms.johnsen_quan import get_johnsen_epochs_arr, plot_epochs
from paradigms.johnsen_quan import compute_log_band_power_avg, compute_z_scores_for_bands
matplotlib.use('Qt5Agg')



In [ ]:
config_path = '../configs/johnsen_cfg_test.yml'
with open(config_path, encoding='utf-8') as f:
    config = yaml.load(f, Loader=yaml.FullLoader)

eeg_file = os.listdir(config['raw_path'])[0]
eeg_full_path = os.path.join(config['raw_path'], eeg_file)
_, raw = load_eeg(eeg_full_path, config)
# dc_channel = get_dc_channel(raw, config['dc_threshold'])
start_time, end_time = get_eeg_timestamps(raw)
ptc_df, patient_id = load_stimulus(config['protocol_path'], start_time, end_time)
patient_trial = ptc_df.loc[(ptc_df['patient_id'] == patient_id) & (ptc_df['trial_type']==config['trial_type'])]
patient_trial['start_sec'] = patient_trial.apply(lambda x: trial_start_sec(x, start_time), axis=1)
patient_trial['end_sec'] = patient_trial.apply(lambda x: trial_end_sec(x, start_time), axis=1)

signal_start_times = []
signal_start_samples = []
for start_sec in patient_trial['start_sec']:
    signal_start_sample, signal_start_time = detect_signal_start(raw, start_sec)
    signal_start_times.append(signal_start_time)
    signal_start_samples.append(signal_start_sample)
    

patient_trial['signal_start_time'] = signal_start_times
patient_trial['signal_start_sample'] = signal_start_samples

if config['verbose']:
    print(raw.info)

events = []
for id, row in patient_trial.iterrows():
    events.extend(get_johnsen_epochs_arr(row['start_time'], row['end_time'], row['signal_start_time'], start_time, config['sfreq']))
epochs = mne.Epochs(raw, events, tmin=0, tmax=2, baseline=None, preload=True)
active_epochs = mne.Epochs(raw, events, event_id=1, tmin=0, tmax=2, baseline=None, preload=True)
reference_epochs = mne.Epochs(raw, events, event_id=0, tmin=0, tmax=2, baseline=None, preload=True)

if config['verbose']:
    plot_epochs(events, start_time, config['sfreq'], row)


### Statistical Analysis

### Logistic Regression

In [39]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
import numpy as np

# Step 1: Prepare the data
def prepare_features(log_band_power_avg):
    # Combine log band power for all bands into a single feature matrix
    features = np.hstack([log_band_power_avg[band][:, np.newaxis] for band in freq_bands.keys()])
    return features

# Prepare features for reference and active epochs
ref_features = prepare_features(ref_log_band_power_avg)
active_features = prepare_features(active_log_band_power_avg)

# Create labels: 0 for reference epochs, 1 for active epochs
labels = np.concatenate([np.zeros(len(ref_features)), np.ones(len(active_features))])

# Combine features and labels
# X = np.vstack([ref_features, active_features])
X = active_features #???

# TODO: Create the label for logistic regression
y = labels

In [ ]:


# Step 2: Standardize the active epochs using the reference epochs
scaler = StandardScaler()
X[:len(ref_features)] = scaler.fit_transform(X[:len(ref_features)])  # Standardize reference epochs
X[len(ref_features):] = scaler.transform(X[len(ref_features):])  # Standardize active epochs

# Step 3: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Step 4: Train the logistic regression model
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(X_train, y_train)

# Step 5: Evaluate the model
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]  # Probabilities for class 1

# Classification metrics
print("Classification Report:")
print(classification_report(y_test, y_pred))

# ROC AUC score
roc_auc = roc_auc_score(y_test, y_proba)
print(f"ROC AUC Score: {roc_auc:.2f}")


### Non-parametric bootstrap

In [ ]:
def bootstrap_roc(z_score_results, true_labels, n_bootstrap=1000, random_state=None):
    """
    Perform non-parametric bootstrap to calculate ROC statistics.

    Parameters:
    - z_score_results: dict, Z-score results for each frequency band.
    - true_labels: array-like, True labels for the epochs.
    - n_bootstrap: int, Number of bootstrap samples.
    - random_state: int or None, Random state for reproducibility.

    Returns:
    - bootstrap_roc_stats: dict, Bootstrap ROC statistics for each frequency band.
    """
    if random_state is not None:
        np.random.seed(random_state)

    bootstrap_roc_stats = {band: {'auc': []} for band in z_score_results.keys()}

    for _ in range(n_bootstrap):
        # Generate bootstrap sample indices
        bootstrap_indices = np.random.choice(len(true_labels), size=len(true_labels), replace=True)
        bootstrap_true_labels = true_labels[bootstrap_indices]

        for band, results in z_score_results.items():
            bootstrap_z_scores = results['z_scores'][bootstrap_indices]

            # Compute ROC curve
            fpr, tpr, _ = roc_curve(bootstrap_true_labels, bootstrap_z_scores)

            # Calculate AUC
            roc_auc = auc(fpr, tpr)

            # Store the AUC for this bootstrap sample
            bootstrap_roc_stats[band]['auc'].append(roc_auc)

    # Calculate mean and 95% confidence interval for AUC
    for band in bootstrap_roc_stats.keys():
        auc_values = np.array(bootstrap_roc_stats[band]['auc'])
        mean_auc = np.mean(auc_values)
        ci_lower = np.percentile(auc_values, 2.5)
        ci_upper = np.percentile(auc_values, 97.5)
        bootstrap_roc_stats[band]['mean_auc'] = mean_auc
        bootstrap_roc_stats[band]['ci_lower'] = ci_lower
        bootstrap_roc_stats[band]['ci_upper'] = ci_upper

    return bootstrap_roc_stats

# Example usage
bootstrap_stats = bootstrap_roc(z_score_results, true_labels, n_bootstrap=1000, random_state=42)

# Print the results
for band, stats in bootstrap_stats.items():
    print(f"\nBand: {band}")
    print(f"Mean AUC: {stats['mean_auc']:.2f}")
    print(f"95% CI: [{stats['ci_lower']:.2f}, {stats['ci_upper']:.2f}]")